# Office GIS + Employment Processing

This notebook prepares tract-level employment data and spatially apportions jobs to 5-mile and 8-mile trade areas.

**Workflow:** load GIS layers, run allocation logic, validate totals, and export summary CSVs.


In [ ]:
# Uncomment if needed
# %pip install -q geopy


In [ ]:
# Uncomment if needed
# %pip install -q geopandas


In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# 1. TOOL: Robust Address Lookup
address_string = "4170 E Ponce de Leon Ave NE, Clarkston, GA 30021"
geolocator = Nominatim(user_agent="clarkston_market_analysis_tool")

try:
    location = geolocator.geocode(address_string, timeout=10)
    if location:
        lat, lon = location.latitude, location.longitude
        print(f"Tool successfully found location: {lat}, {lon}")
    else:
        lat, lon = 33.8055, -84.2373 
except (GeocoderTimedOut, GeocoderServiceError):
    lat, lon = 33.8055, -84.2373

# 2. LOAD DATA (Fixed for Excel Tabs!)
tracts_path = '../data/Office/ARC_Census_Tracts_2614808396197599278'
jobs_path = '../data/Office/arc_employment.xlsx'

tracts = gpd.read_file(tracts_path)

# Load the specific tabs from the single Excel file
print("Loading Excel tabs...")
df20 = pd.read_excel(jobs_path, sheet_name='Emp2020_byTract2010')
df30 = pd.read_excel(jobs_path, sheet_name='Emp2030_byTract2010')
df40 = pd.read_excel(jobs_path, sheet_name='Emp2040_byTract2010')

# Merge all three years into one giant dataframe
# We drop 'County' from 2030 and 2040 so it doesn't duplicate in the merge
jobs = df20.merge(df30.drop(columns=['County']), on='Tract2010')
jobs = jobs.merge(df40.drop(columns=['County']), on='Tract2010')

# 3. CLEAN DATA
jobs.columns = jobs.columns.str.strip()
# Get all columns that represent job data (N codes and Total)
sector_cols = [c for c in jobs.columns if c.startswith('N') or c == 'Total']

for col in sector_cols:
    jobs[col] = pd.to_numeric(jobs[col].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

# Join jobs to shapes
jobs['Tract2010'] = jobs['Tract2010'].astype(str)
tracts['GEOID10'] = tracts['GEOID10'].astype(str)
merged = tracts.merge(jobs, left_on='GEOID10', right_on='Tract2010')

# 4. PROJECT TO METERS
merged = merged.to_crs(epsg=26916)
center = gpd.GeoSeries([Point(lon, lat)], crs=4326).to_crs(epsg=26916).iloc[0]

# 5. SPATIAL APPORTIONMENT LOGIC
def calculate_radius_jobs(radius_miles, data_gdf, cols_to_multiply):
    radius_meters = radius_miles * 1609.34
    buffer = center.buffer(radius_meters)
    
    subset = data_gdf[data_gdf.geometry.intersects(buffer)].copy()
    
    subset['total_area'] = subset.geometry.area
    subset['intersect_area'] = subset.geometry.intersection(buffer).area
    subset['weight'] = subset['intersect_area'] / subset['total_area']
    
    for col in cols_to_multiply:
        subset[col] = subset[col] * subset['weight']
        
    return subset[cols_to_multiply].sum()

print("Calculating 5-mile and 8-mile radii...")
# Run the raw calculations
raw_5mi = calculate_radius_jobs(5, merged, sector_cols)
raw_8mi = calculate_radius_jobs(8, merged, sector_cols)

# 6. RESHAPE AND MAP SECTORS
# Official NAICS Title Mapping
naics_map = {
    'N11': 'Agriculture, Forestry, Fishing & Hunting',
    'N21': 'Mining, Quarrying, Oil & Gas',
    'N22': 'Utilities',
    'N23': 'Construction',
    'N313233': 'Manufacturing',
    'N42': 'Wholesale Trade',
    'N4445': 'Retail Trade',
    'N4849': 'Transportation & Warehousing',
    'N51': 'Information',
    'N52': 'Finance & Insurance',
    'N53': 'Real Estate, Rental & Leasing',
    'N54': 'Professional, Scientific & Tech Services',
    'N55': 'Management of Companies',
    'N56': 'Administrative & Waste Management',
    'N61': 'Educational Services',
    'N62': 'Health Care & Social Assistance',
    'N71': 'Arts, Entertainment & Recreation',
    'N72': 'Accommodation & Food Services',
    'N81': 'Other Services',
    'N92': 'Public Administration',
    'Total': 'Total Employment'
}

def format_final_report(raw_series):
    # Convert series to dataframe
    df = raw_series.reset_index()
    df.columns = ['Code', 'Jobs']
    
    # Extract the base code (e.g., 'N11_20' -> 'N11', 'Total' -> 'Total')
    df['Base_Code'] = df['Code'].apply(lambda x: x.split('_')[0] if '_' in x else x)
    
    # Extract the year (e.g., 'N11_20' -> '2020')
    def get_year(code):
        if code == 'Total': return '2020' 
        if '_20' in code: return '2020'
        if '_30' in code: return '2030'
        if '_40' in code: return '2040'
        return 'Unknown'
    
    df['Year'] = df['Code'].apply(get_year)
    
    # Map the real names
    df['Sector'] = df['Base_Code'].map(naics_map)
    
    # Pivot the table so Years become columns
    pivot_df = df[df['Base_Code'] != 'Total'].pivot(index='Sector', columns='Year', values='Jobs').reset_index()
    
    # Calculate totals dynamically so they align with the columns
    total_row = pd.DataFrame({
        'Sector': ['Total Employment'],
        '2020': [pivot_df['2020'].sum()],
        '2030': [pivot_df['2030'].sum()],
        '2040': [pivot_df['2040'].sum()]
    })
    
    # Combine and round
    final_df = pd.concat([pivot_df, total_row], ignore_index=True)
    final_df[['2020', '2030', '2040']] = final_df[['2020', '2030', '2040']].round(0).astype(int)
    
    # Optional: Sort by largest 2020 employment for better readability
    final_df = final_df.sort_values(by='2020', ascending=False).reset_index(drop=True)
    
    return final_df

# Generate Final DataFrames
df_5mi = format_final_report(raw_5mi)
df_8mi = format_final_report(raw_8mi)

print("\n--- 5 MILE RADIUS ---")
print(df_5mi.to_string(index=False))
print("\n--- 8 MILE RADIUS ---")
print(df_8mi.to_string(index=False))

# Export to CSV for your report
df_5mi.to_csv('../data/Office/Employment_5Mile_Radius.csv', index=False)
df_8mi.to_csv('../data/Office/Employment_8Mile_Radius.csv', index=False)
print("\nSuccess! Files '../data/Office/Employment_5Mile_Radius.csv' and '../data/Office/Employment_8Mile_Radius.csv' have been saved in your current folder.")

Loading Excel tabs...
Calculating 5-mile and 8-mile radii...

--- 5 MILE RADIUS ---
                                  Sector   2020   2030   2040
                        Total Employment 145953 163304 170155
         Health Care & Social Assistance  28464  34330  37901
                    Educational Services  23039  26094  27389
                            Retail Trade  13427  12436  11835
                   Public Administration  11836  12540  13204
       Administrative & Waste Management  10902  12591  13387
           Accommodation & Food Services   8168   8989   8517
Professional, Scientific & Tech Services   7416   8521   9257
                            Construction   5925   8095   8731
                           Manufacturing   5662   5564   5508
                          Other Services   5587   6572   6034
                     Finance & Insurance   4975   5218   5373
                         Wholesale Trade   4890   5365   5708
            Transportation & Warehousing   4028 

In [2]:
# 5. SPATIAL APPORTIONMENT & ROBUST QA CHECKS
def calculate_radius_jobs(radius_miles, data_gdf, cols_to_multiply):
    radius_meters = radius_miles * 1609.34
    buffer = center.buffer(radius_meters)
    
    # Get tracts that touch our circle
    subset = data_gdf[data_gdf.geometry.intersects(buffer)].copy()
    
    # Calculate Weights
    subset['total_area'] = subset.geometry.area
    subset['intersect_area'] = subset.geometry.intersection(buffer).area
    subset['weight'] = subset['intersect_area'] / subset['total_area']
    
    # --- 🛑 ROBUST DATA CHECKS START HERE 🛑 ---
    print(f"\n[ RUNNING QA CHECKS FOR {radius_miles}-MILE RADIUS ]")
    
    # Check 1: Weight Bounds
    max_w, min_w = subset['weight'].max(), subset['weight'].min()
    if max_w > 1.0001 or min_w < 0:
        print(f"❌ ERROR: Geometry weights out of bounds (Min: {min_w:.2f}, Max: {max_w:.2f}). Check projections!")
    else:
        print(f"✅ PASS: All {len(subset)} tracts correctly weighted between 0% and 100%.")

    # Check 2: Negative Numbers
    negatives = (subset[cols_to_multiply] < 0).sum().sum()
    if negatives > 0:
        print(f"❌ ERROR: Found {negatives} negative job values!")
    else:
        print("✅ PASS: No negative job counts detected.")
        
    # Check 3: Regional Sanity Check (Using 2020 Total)
    # We check if the jobs in the radius are less than the whole map
    regional_total = data_gdf['Total_x'].sum() if 'Total_x' in data_gdf.columns else data_gdf['Total'].sum()
    radius_total = (subset['Total_x'] * subset['weight']).sum() if 'Total_x' in subset.columns else (subset['Total'] * subset['weight']).sum()
    
    if radius_total > regional_total:
        print(f"❌ ERROR: Apportioned jobs ({radius_total:,.0f}) exceed total regional jobs ({regional_total:,.0f})!")
    else:
        print(f"✅ PASS: Radius jobs ({radius_total:,.0f}) are a logical subset of regional jobs ({regional_total:,.0f}).")
    # --- 🛑 ROBUST DATA CHECKS END HERE 🛑 ---

    # Do the actual multiplication
    for col in cols_to_multiply:
        subset[col] = subset[col] * subset['weight']
        
    return subset[cols_to_multiply].sum()

print("\nExecuting calculations and audits...")
raw_5mi = calculate_radius_jobs(5, merged, sector_cols)
raw_8mi = calculate_radius_jobs(8, merged, sector_cols)


Executing calculations and audits...

[ RUNNING QA CHECKS FOR 5-MILE RADIUS ]
✅ PASS: All 67 tracts correctly weighted between 0% and 100%.
✅ PASS: No negative job counts detected.
✅ PASS: Radius jobs (145,953) are a logical subset of regional jobs (2,633,280).

[ RUNNING QA CHECKS FOR 8-MILE RADIUS ]
✅ PASS: All 168 tracts correctly weighted between 0% and 100%.
✅ PASS: No negative job counts detected.
✅ PASS: Radius jobs (377,061) are a logical subset of regional jobs (2,633,280).
